# NM03 Sieci rekurencyjne: RNN, LSTM, GRU i ich odmiany

**Motywacja:** Jak przewidywać wartości na podstawie **sekwencji** obserwacji? Jak przechowywać informacje potrzebne wiele kroków później?

**Zakres notebooka**

1. Przekształcenie szeregu czasowego na dane użyteczne w metodach uczenia maszynowego
2. Rekurencyjna sieć neuronowa (RNN), współdzielenie parametrów w czasie i propagacja wsteczna przez czas (BPTT).
3. Motywacja, równania i działanie pamięci długotrwałej i krótkotrwałej (LSTM) oraz bramek rekurencyjnych (GRU).
4. Porównanie RNN, LSTM i GRU.
5. Sieci dwukierunkowe oraz inne zaawansowane architektury.
6. Przewidywanie wieloetapowe, analiza błędów i ograniczeń porównań.

**Oznaczenia:** 

$B$ - liczba sekwencji w danych, 

$T$ - liczba kroków w sekwencji, 

$F$ - liczba cech obserwacji, 

$H$ - wymiar stanu ukrytego (hidden state). 

W notebooku używamy `batch_first=True`, dlatego wejście ma kształt `(B, T, F)`, a pojedyncza prognoza dla każdej sekwencji - `(B, 1)`.


Ćwiczenia 1-4 obejmują zagadnienia wprowadzające omawiane na zajęciach. Ćwiczenia 5-15 są dodatkowymi zadaniami do samodzielnego wykonania.

W komórkach kodu oznaczonych `TODO` należy uzupełnić brakujące linie. Po napisaniu rozwiązania usuń odpowiednią instrukcję `raise NotImplementedError(...)`. Wykonuj komórki po kolei i sprawdzaj wyniki asercji.

Do wykonania ćwiczeń potrzebne są biblioteki `numpy`, `matplotlib` i `torch`. Kod można uruchamiać na CPU. Wyniki treningu mogą się nieznacznie różnić zależnie od środowiska uruchomieniowego.

Modele należy wybierać na podstawie walidacji; zbiór testowy jest wykorzystywany dopiero w ćwiczeniu 8.


**Słowniczek:** 

*stan ukryty* $h_t$ jest reprezentacją informacji z dotychczasowych chwil; 

*stan komórki* $c_t$ stanowi dodatkową pamięć w Long Short-Term Memory (LSTM); 

*bramka* jest obliczaną z danych maską o wartościach od 0 do 1; 

*BPTT* backpropagation through time (propagacja wsteczna przez czas);

*gradient clipping* ucinanie zbyt dużego gradientu; zob. dokumentacja https://docs.pytorch.org/docs/stable/generated/torch.nn.utils.clip_grad_norm_.html; 

*długość okna* (*lookback*) parametr mówiący, ile przeszłych pomiarów trafia do modelu przy przewidywaniu kolejnego kroku.


## Co widzi model?

Dla jednego szeregu w oknie znajdzie się $T$ następujących po sobie obserwacji; 

`F=1`, bo każda chwila zawiera jedną liczbę. 

Jeden batch 8 okien ma więc wymiar `(8, T, 1)`.

**Sekwencje są niezależnymi przykładami w batchu** - nie należy mylić osi batcha z osią czasu.

W kolejnych przykładach używamy `batch_first=True`. 

W sieciach konwolucyjnych modelowaliśmy dane przesuwając filtr po współrzędnych obrazu; tutaj te same wagi komórki Recurrent Neural Network (RNN) przesuwają się po czasie.


## 0. Importy

Ustawiamy ziarna generatorów liczb pseudolosowych, aby ułatwić odtwarzanie doświadczeń. Wyniki mogą się nieznacznie różnić w zależności od wersji biblioteki lub urządzenia. Do zajęć wystarcza CPU; jeśli dostępny jest akcelerator CUDA, można go wykorzystać automatycznie.


In [ ]:
import copy
import math
import random
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

SEED = 12
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_num_threads(min(2, torch.get_num_threads()))
def count_parameters(model):
    """Liczba parametrów modelu podlegających uczeniu (łącznie z biasami)."""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


print("PyTorch:", torch.__version__, "| urządzenie:", DEVICE)


## 1. Szereg czasowy jako zbiór sekwencji

W zadaniach z NM01 każdy punkt danych mogliśmy rozpatrywać osobno. W NM02 relacje między pikselami odwzorowywały filtry przesuwające się po obrazie. W szeregu czasowym znaczenie ma **kolejność pomiarów**: obecna wartość może zależeć od historii, której nie da się ująć w jednej obserwacji.

Rozpatrujemy zadanie **wiele-do-jednego** (*many-to-one*): na podstawie $T$ kolejnych pomiarów prognozujemy następny. Przygotowany szereg zawiera okresy 24 i 90 kroków, na których można dostrzec pewien trend. Dane są lekko zaszumione.

W tym zadaniu skupiamy się na sposobie przetwarzania sekwencji.


In [ ]:
t = np.arange(900, dtype=np.float32)
rng = np.random.default_rng(SEED)
series = (
    0.9 * np.sin(2 * np.pi * t / 24)
    + 0.4 * np.sin(2 * np.pi * t / 90)
    + 0.001 * t
    + rng.normal(0, 0.08, size=len(t))
).astype(np.float32)

plt.figure(figsize=(11, 3.2))
plt.plot(t, series, lw=1, label="pomiar")
plt.axvline(600, color="tab:orange", ls="--", label="początek walidacji")
plt.axvline(750, color="tab:red", ls="--", label="początek testu")
plt.xlabel("Krok czasowy")
plt.ylabel("Wartość")
plt.title("Syntetyczny szereg - kolejnych pomiarów NIE mieszamy")
plt.legend(loc="upper left")
plt.tight_layout()
plt.show()


### Ćwiczenie 1. Tworzenie okien

Rozważ szereg $[0,1,2,3,4]$. Dla długości okna równej 3 dostajemy pary $[0,1,2]\to3$ i $[1,2,3]\to4$.

Przeanalizuj funkcję `make_windows`, a następnie:

1. wyprowadź liczbę przykładów dla szeregu długości $N$ i okna długości $T$;
2. sprawdź rozmiary tensorów wyjściowych oraz zawartość pierwszego okna;
3. wyjaśnij, co łączy dwa sąsiadujące okna i dlaczego ma to znaczenie dla podziału danych.


In [ ]:
def make_windows(values, lookback):
    """Z szeregu (N,) tworzy X:(N-lookback, lookback, 1), y:(N-lookback, 1)."""
    values = np.asarray(values, dtype=np.float32).reshape(-1)
    if not (1 <= lookback < len(values)):
        raise ValueError("lookback musi być >=1 i mniejsze od długości szeregu")
    X = np.stack([values[i : i + lookback] for i in range(len(values) - lookback)])
    y = values[lookback:]
    return torch.from_numpy(X[..., None]), torch.from_numpy(y[:, None])


X_demo, y_demo = make_windows(np.arange(10), 3)
assert X_demo.shape == (7, 3, 1) and y_demo.shape == (7, 1)
assert X_demo[0, :, 0].tolist() == [0, 1, 2]
assert y_demo[0, 0].item() == 3
print("Pierwsze okno:", X_demo[0, :, 0].tolist(), "->", y_demo[0, 0].item())


**Pytania do ćwiczenia 1**

1. Dlaczego otrzymujemy dokładnie $N-T$ przykładów, a nie $N$?
2. Co mogłoby się stać przy losowym podziale silnie nakładających się okien?


### Podział chronologiczny i wyciek informacji (data leakage)

Ze swojej natury sąsiednie okna zawierają wiele podobnych obserwacji. Losowe dzielenie takich okien na trening i test mogłoby prowadzić do wycieku informacji: przykłady treningowe obejmowałyby okres późniejszy niż oceniane prognozy.

Dlatego definiujemy trzy kolejne przedziały czasu: trening `[0, 600)`, walidację `[600, 750)` i test `[750, 900)`. Średnią $\mu$ oraz odchylenie standardowe $\sigma$ obliczamy **tylko dla treningu** i używamy ich do przekształcenia całego szeregu według wzoru $(x-\mu)/\sigma$.

O przynależności przykładu do zbioru decyduje **czas przewidywanej wartości**. Pierwsze okno walidacyjne może więc zawierać pomiary z końca treningu: są one już znane w chwili przewidywania. Niedozwolone byłoby wykorzystanie przyszłych wartości albo statystyk wyznaczonych na danych testowych.


In [ ]:
LOOKBACK = 36
train_end, val_end = 600, 750
mu = float(series[:train_end].mean())
sigma = float(series[:train_end].std())
assert sigma > 0
scaled = (series - mu) / sigma
X_all, y_all = make_windows(scaled, LOOKBACK)
target_indices = np.arange(LOOKBACK, len(series))

train_mask = target_indices < train_end
val_mask = (target_indices >= train_end) & (target_indices < val_end)
test_mask = target_indices >= val_end
Xtr, ytr = X_all[train_mask], y_all[train_mask]
Xva, yva = X_all[val_mask], y_all[val_mask]
Xte, yte = X_all[test_mask], y_all[test_mask]

print("Train / val / test:", Xtr.shape, Xva.shape, Xte.shape)
print("Pierwsza walidacja - kroki wejścia:", (600 - LOOKBACK, 599), "cel:", 600)
assert len(Xtr) + len(Xva) + len(Xte) == len(series) - LOOKBACK
assert len(Xva) == 150 and len(Xte) == 150
assert abs(float(scaled[:train_end].mean())) < 1e-5


### Ćwiczenie 2. Proste metody prognozowania

Zanim wytrenujemy sieć, potrzebujemy naiwnych metod odniesienia (baselinów). Niech pierwsza metoda zakłada, że następny pomiar będzie równy ostatniemu znanemu, czyli po prostu: $\hat y_t=x_{t-1}$.

Porównamy średni błąd bezwzględny (MAE) i pierwiastek błędu średniokwadratowego (RMSE):

$$\mathrm{MAE}=\frac{1}{N}\sum_i |\hat y_i-y_i|,\qquad
\mathrm{RMSE}=\sqrt{\frac{1}{N}\sum_i(\hat y_i-y_i)^2}.$$

Drugą metodą jest prognoza sezonowa (season forecasting), wykorzystująca okres 24 kroków: $\hat y_t=x_{t-24}$. Oba wyniki porównamy na walidacji. W tej demonstracji okres 24 znamy z konstrukcji danych; na rzeczywistym szeregu należałoby ustalić go wyłącznie z danych treningowych lub wiedzy dziedzinowej, np. 365 dla dziennych zapisów, 60 dla minutowych, itd.


In [ ]:
def metrics(pred, target):
    pred, target = pred.detach().cpu(), target.detach().cpu()
    return {
        "MAE": (pred - target).abs().mean().item(),
        "RMSE": ((pred - target).square().mean().sqrt()).item(),
    }


baseline_va = Xva[:, -1, :]
baseline_te = Xte[:, -1, :]
print("Metoda ostatniej wartości - walidacja:", metrics(baseline_va, yva))
# Wynik na teście obliczymy dopiero w ćwiczeniu 8, po wyborze modelu na walidacji.


### Metoda sezonowa i interpretacja błędów

Ponieważ okno obejmuje 36 obserwacji, wartość sprzed 24 kroków znajduje się pod indeksem `X[:, -24, :]`. Błędy w komórkach kodu liczymy na skali znormalizowanej. Aby wyrazić MAE lub RMSE w pierwotnych jednostkach, mnożymy je przez dodatnie $\sigma$; dodanie średniej $\mu$ nie zmienia różnic między prognozą a obserwacją.


In [ ]:
SEASON = 24
assert LOOKBACK >= SEASON
seasonal_va = Xva[:, -SEASON, :]
seasonal_te = Xte[:, -SEASON, :]
for name, p in [("Ostatnia obserwacja", baseline_va), ("Metoda sezonowa", seasonal_va)]:
    e = metrics(p, yva)
    print(f'{name:21} | val MAE={e["MAE"]:.3f} | val RMSE={e["RMSE"]:.3f}')


**Pytania do ćwiczenia 2**

1. Która metoda odniesienia osiąga niższy RMSE na walidacji i dlaczego?
2. Czym różni się interpretacja MAE od RMSE?


## 2. Sieć rekurencyjna (RNN)

### Dlaczego powstała ta architektura?

Model z gęstymi warstwami neuronowymi może przyjąć okno o ustalonej długości jako pojedynczy wektor. Nie zawiera jednak w swojej konstrukcji reguły, która w podobny sposób przetwarzałaby każdy kolejny pomiar. **RNN rozwiązuje problem współdzielenia sposobu przetwarzania w czasie**: tę samą transformację stosujemy dla dowolnej liczby kroków, a między krokami przekazujemy stan ukryty.

Dla komórki z aktywacją $\tanh$ aktualizacja ma postać

$$h_t=\tanh(W_xx_t+W_hh_{t-1}+b),\qquad h_0=0.$$

Przyjmujemy $x_t\in\mathbb{R}^F$, $h_t\in\mathbb{R}^H$, $W_x\in\mathbb{R}^{H\times F}$ i $W_h\in\mathbb{R}^{H\times H}$. Stan $h_t$ jest reprezentacją dotychczasowej historii, a jego wymiar pozostaje stały niezależnie od długości sekwencji. Jest to zaleta obliczeniowa, ale zarazem ograniczenie: odległe informacje mogą zanikać w trakcie aktualizacji.

Jeden z pierwszych mechanizmów rekurencyjnego stanu opisuje m.in. praca Jeffreya Elmana z 1990 r. [1].


### Ćwiczenie 3. RNN od podstaw

Zapisz w Pythonie równanie z poprzedniej komórki. Zwróć uwagę na `x[:, step, :]`: to **wszystkie** elementy batcha, ale **jeden** krok sekwencji. Wagi są te same w każdej iteracji pętli.

1. Sprawdź kształty zwracanych stanów.
2. Zmień liczbę kroków wejściowych z 6 na 12, czy zmieni się liczba parametrów?
3. Uruchom `backward()`: czy waga `h2h` otrzymuje gradient?

**Do uzupełnienia:** jedna linia aktualizacji stanu w `ManualRNN.forward`.


In [ ]:
class ManualRNN(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.x2h = nn.Linear(input_size, hidden_size, bias=True)
        self.h2h = nn.Linear(hidden_size, hidden_size, bias=False)
        self.hidden_size = hidden_size

    def forward(self, x):
        B, T, _ = x.shape
        h = x.new_zeros(B, self.hidden_size)
        states = []
        for step in range(T):
            # TODO: uaktualnij h według wzoru RNN (tanh, x2h, h2h).
            raise NotImplementedError("Uzupełnij kod: aktualizacja stanu RNN")
            states.append(h)
        return torch.stack(states, dim=1), h


manual = ManualRNN(1, 8)
seq, last = manual(torch.randn(4, 6, 1))
assert seq.shape == (4, 6, 8) and last.shape == (4, 8)
seq.sum().backward()
assert manual.h2h.weight.grad is not None
print("RNN działa; norma gradientu:", manual.h2h.weight.grad.norm().item())


**Pytania do ćwiczenia 3**

1. Dlaczego liczba parametrów RNN nie zależy od długości sekwencji?
2. Jak sprawdzenie gradientu `h2h` potwierdza, że działa propagacja wsteczna?


## 3. Propagacja wsteczna przez czas (BPTT)

W RNN graf obliczeń składa się wzdłuż czasu. Macierz $W_h$ jest wykorzystywana w kolejnych aktualizacjach, więc gradient jej parametrów uwzględnia wkład wszystkich tych kroków.

Dla prostej rekurencji liniowej $h_t=ah_{t-1}$ otrzymujemy

$$\frac{\partial h_T}{\partial h_0}=a^T.$$

Gdy $|a|<1$, pochodna maleje wraz z $T$, a gdy $|a|>1$, może gwałtownie rosnąć. W rzeczywistej sieci gradient zależy od iloczynów macierzy Jacobiego i pochodnych aktywacji, dlatego powyższy wzór jest tylko ilustracją problemu. 

Trudność uczenia długich zależności badali m.in. Bengio, Simard i Frasconi [2].


### Ćwiczenie 4. Zanik i eksplozja gradientu

Narysuj $|a|^T$ dla $a=0{,}8$, $1$ i $1{,}1$ w skali logarytmicznej. Co stanie się z odległą informacją po 50 krokach? Następnie sprawdź wynik **autograd** na liniowej rekurencji: czy pochodna obliczona przez PyTorch zgadza się z analitycznym $a^T$?

**Do uzupełnienia:** jedna linia odpowiedzialna za wykres.


In [ ]:
steps = np.arange(1, 51)
plt.figure(figsize=(8, 3.2))
for a in [0.8, 1.0, 1.1]:
    # TODO: narysuj |a|^T w skali logarytmicznej (plt.semilogy).
    raise NotImplementedError("Uzupełnij kod: wykres gradientu")
plt.xlabel("Liczba kroków T")
plt.ylabel(r"$|\partial h_T / \partial h_0|$")
plt.title("Długi iloczyn lokalnych pochodnych")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()
assert 0.8**50 < 1e-4 and 1.1**50 > 100


### Autograd naprawdę sumuje wpływ wielu kroków

Na rysunku użyliśmy wzoru matematycznego. Poniżej wykonujemy tę samą rekurencję w PyTorch i liczymy gradient przez **rzeczywisty graf obliczeń**. Jeśli $h_0$ wymaga gradientu, to `torch.autograd.grad(h_T, h_0)` powinien zwrócić $a^T$.

**Dodatkowy wariant:** zamiast `h = a*h` wstaw `h = torch.tanh(a*h)`. Czy nadal otrzymasz $a^T$? Wyjaśnij dodatkowy czynnik pochodzący od `tanh`.


In [ ]:
for a, steps_count in [(0.8, 20), (1.1, 20)]:
    h0 = torch.tensor(1.0, requires_grad=True)
    h = h0
    for _ in range(steps_count):
        h = a * h
    grad = torch.autograd.grad(h, h0)[0].item()
    print(f"a={a}: autograd={grad:.6f}, teoria={a**steps_count:.6f}")
    assert np.isclose(grad, a**steps_count, rtol=1e-5)


**Pytania do ćwiczenia 4**

1. Co dzieje się z gradientem przy $|a|<1$ i $|a|>1$?
2. Dlaczego ograniczanie normy gradientu nie naprawia zanikających gradientów?


**Zadania dodatkowe:** ćwiczenia 5-15 można wykonać samodzielnie.

## 4. Architektury z bramkami: LSTM i GRU

### LSTM jako połączenie długotrwałej oraz krótkotrwałej pamięci

**Problem:** prosta architektura RNN wielokrotnie przekształca cały stan, przez co uczenie zależności rozdzielonych wieloma krokami bywa trudne.

**Motywacja LSTM** polega na oddzieleniu stanu ukrytego $h_t$ od stanu komórki $c_t$, którego aktualizacja zawiera składnik addytywny. Wyuczone bramki decydują, ile dawnej informacji zachować oraz ile nowej wprowadzić.

Dla standardowej wersji LSTM z bramką zapominania zachodzi:

$$\begin{aligned}
i_t&=\sigma(W_i[x_t,h_{t-1}]+b_i),\quad
f_t=\sigma(W_f[x_t,h_{t-1}]+b_f),\\
o_t&=\sigma(W_o[x_t,h_{t-1}]+b_o),\quad
g_t=\tanh(W_g[x_t,h_{t-1}]+b_g),\\
c_t&=f_t\odot c_{t-1}+i_t\odot g_t,\\
h_t&=o_t\odot\tanh(c_t).
\end{aligned}$$

Symbol $\odot$ oznacza mnożenie element po elemencie, a $\sigma$ jest sigmoidą. Bramki wejścia $i_t$, zapominania $f_t$ i wyjścia $o_t$ umożliwiają selektywną aktualizację pamięci. Suma w równaniu dla $c_t$ ułatwia przekazywanie informacji przez wiele chwil, lecz nie gwarantuje niezanikania gradientu w każdym zadaniu.

LSTM opisali Sepp Hochreiter i Jürgen Schmidhuber w 1997 r. [3]. Bramkę zapominania zaproponowali później Gers, Schmidhuber i Cummins [4].


### GRU jako mniej złożona pamięć z bramkami

**Problem:** LSTM korzysta z osobnego stanu komórki i trzech bramek, co zwiększa liczbę parametrów.

**Motywacją GRU** było uproszczenie sterowania pamięcią: komórka ma tylko jeden stan $h_t$ oraz dwie bramki, jedna aktualizacji $z_t$ i druga resetu $r_t$. Może dzięki temu stanowić praktyczną alternatywę, gdy zależy nam na mniejszym modelu.

W konwencji stosowanej przez `torch.nn.GRU`:

$$\begin{aligned}
r_t&=\sigma(W_{ir}x_t+b_{ir}+W_{hr}h_{t-1}+b_{hr}),\\
z_t&=\sigma(W_{iz}x_t+b_{iz}+W_{hz}h_{t-1}+b_{hz}),\\
n_t&=\tanh\bigl(W_{in}x_t+b_{in}+r_t\odot(W_{hn}h_{t-1}+b_{hn})\bigr),\\
h_t&=z_t\odot h_{t-1}+(1-z_t)\odot n_t.
\end{aligned}$$

Duża wartość $z_t$ sprzyja zachowaniu starego stanu, a bramka resetu wpływa na wykorzystanie przeszłości podczas tworzenia kandydata. W oryginalnym sformułowaniu GRU kolejność zastosowania bramki resetu i transformacji liniowej jest inna niż w PyTorch; również konwencja symbolu bramki aktualizacji w publikacjach nie zawsze jest taka sama. Wzory powyżej są zgodne z dokumentacją użytej biblioteki [10].

GRU opisano w 2014 r. w pracy [5].

**Ograniczenie:** mniejsza liczba parametrów nie oznacza automatycznie mniejszego błędu ani krótszego treningu na każdym urządzeniu.


### Miniaturowy przykład działania bramek

Prześledźmy jeden wymiar pamięci, bez uczenia parametrów. Dla LSTM aktualizacja $c_t=f_tc_{t-1}+i_tg_t$ pozwala zachować dawną wartość lub wprowadzić nową. Dla GRU bramka $z_t$ reguluje udział starego stanu w nowym stanie.

W kodzie wartości bramek podajemy ręcznie, aby zobaczyć ich wpływ. W wytrenowanej sieci bramki są obliczane z bieżącego wejścia i poprzedniego stanu.


In [ ]:
old_c, candidate, input_gate = 2.0, 0.5, 0.5
for forget_gate in [0.1, 0.9]:
    new_c = forget_gate * old_c + input_gate * candidate
    print(f"LSTM | zapominanie={forget_gate:.1f} -> pamięć={new_c:.2f}")
old_h, candidate_h = 2.0, -1.0
for update_gate in [0.1, 0.9]:
    new_h = update_gate * old_h + (1 - update_gate) * candidate_h
    print(f"GRU  | aktualizacja={update_gate:.1f} -> stan={new_h:.2f}")


### Ćwiczenie 5. Ile parametrów mają RNN, GRU i LSTM? (zadanie matematyczne)

Przyjmij **wymiar wejścia $F=1$**, **wymiar stanu ukrytego $H=24$** i jedną warstwę rekurencyjną, bez dwukierunkowości. Każdy model prognozuje jedną wartość przez liniową głowicę $\mathbb{R}^{H}\to\mathbb{R}$.

1. **Wyprowadź wzory** na liczbę parametrów pojedynczej warstwy `nn.RNN`, `nn.GRU` i `nn.LSTM`. Każda bramka zawiera macierz wag wejścia $H\times F$, macierz wag rekurencyjnych $H\times H$ i **dwa** wektory biasu o długości $H$ (tak PyTorch przechowuje parametry). RNN ma jedną transformację, GRU - trzy, LSTM - cztery.
2. Oblicz liczby parametrów dla $F=1$, $H=24$, **oddzielnie dla warstwy i dla całego modelu z głowicą**. Ile parametrów ma sama głowica?
3. Uzupełnij trzy wzory w kodzie poniżej i sprawdź zgodność z `count_parameters()` dla warstw PyTorch. Zwróć uwagę, że **ręczna `ManualRNN` ma tylko jeden bias**, więc jej liczba parametrów będzie inna niż w `nn.RNN`.
4. Uzasadnij, **dlaczego liczba parametrów nie zależy od długości sekwencji $T$**, chociaż dłuższa sekwencja oznacza więcej operacji.

**Wskazówka:** przy dwóch biasach PyTorch liczba parametrów jednej transformacji to $HF+H^2+2H$.


In [ ]:
F, H = 1, 24
# TODO: wpisz wzory dla jednej warstwy (bez głowicy).
# Wskazówka: RNN ma jedną transformację, GRU trzy, LSTM cztery.
rnn_core = None  # TODO: podaj wzór dla pojedynczej transformacji
gru_core = None  # TODO: wykorzystaj wzór dla jednej transformacji
lstm_core = None  # TODO: wykorzystaj wzór dla jednej transformacji
assert all(v is not None for v in (rnn_core, gru_core, lstm_core)), (
    "Uzupełnij kod: wpisz trzy wzory na liczbę parametrów"
)

head_params = H + 1  # liniowe przekształcenie R^H -> R, z biasem
expected_core = {"RNN": rnn_core, "GRU": gru_core, "LSTM": lstm_core}
for kind, layer in [
    ("RNN", nn.RNN(F, H, batch_first=True)),
    ("GRU", nn.GRU(F, H, batch_first=True)),
    ("LSTM", nn.LSTM(F, H, batch_first=True)),
]:
    observed = count_parameters(layer)
    assert expected_core[kind] == observed, f"Sprawdź wzór dla {kind}"
    print(f"{kind:4} | warstwa: {observed:4} | z głowicą: {observed + head_params:4}")

# Sprawdzenie liczby parametrów dla każdej warstwy jest w pętli powyżej.


## 5. Porównanie RNN, LSTM i GRU

Modele wykonują to samo zadanie: prognozują następny pomiar na podstawie 36 wcześniejszych. Trenujemy je na jednakowych danych, przy takim samym wymiarze stanu, liczbie epok, metodzie optymalizacji i funkcji straty. Używamy jednej warstwy rekurencyjnej i liniowej głowicy regresji.

Takie porównanie **nie wyrównuje liczby parametrów**: LSTM ma ich więcej niż GRU i prosta RNN przy ustalonym $H$. Nie stanowi też rozstrzygającego rankingu architektur - wyniki zależą od danych, ziarna losowego i ustawień. Wybrany model zostanie oceniony na zbiorze testowym dopiero po zakończeniu wyboru na podstawie walidacji. **Nie zaglądamy do wyników testowych podczas strojenia modelu ani doboru baseline’ów.**


In [ ]:
class Forecaster(nn.Module):
    def __init__(self, kind="GRU", hidden_size=24):
        super().__init__()
        choices = {"RNN": nn.RNN, "GRU": nn.GRU, "LSTM": nn.LSTM}
        if kind not in choices:
            raise ValueError(f"Nieznana architektura: {kind}")
        self.core = choices[kind](1, hidden_size, batch_first=True)
        self.head = nn.Linear(hidden_size, 1)

    def forward(self, x):
        states, _ = self.core(x)  # (B,T,H)
        return self.head(states[:, -1, :])  # (B,1)


for kind in ("RNN", "GRU", "LSTM"):
    m = Forecaster(kind, hidden_size=24)
    assert m(torch.randn(7, LOOKBACK, 1)).shape == (7, 1)
    print(kind, "parametry z głowicą:", count_parameters(m))


### Ćwiczenie 6. Pętla uczenia i walidacji

Z NM01_freestyle.ipynb znamy `loss.backward()` i `optimizer.step()`. Tu składamy je w pełną procedurę:

1. `model.train()` włącz tryb treningowy, przejdź po batchach, oblicz MSE.
2. `optimizer.zero_grad()` -> `loss.backward()` -> `clip_grad_norm_()` -> `optimizer.step()`.
3. `model.eval()` i `torch.no_grad()` - oszacuj RMSE na całej walidacji, sumując błędy po **przykładach**, a nie uśredniając RMSE z minibatchy.
4. Zapamiętaj wagi z epoki o najniższym błędzie walidacyjnym (`state_dict()` wymaga **kopii**, bo parametry zmieniają się w kolejnych epokach).

**Pytania:** 

1. Dlaczego walidacja nie aktualizuje wag?
2. Dlaczego gradient clipping wykonujemy **po** `backward()`, ale **przed** `step()`?

**Do uzupełnienia:** trzy instrukcje uczenia modelu; reszta procedury jest podana.


In [ ]:
def evaluate(model, loader):
    """RMSE w znormalizowanych jednostkach, ważone liczbą przykładów."""
    model.eval()
    squared_error, count = 0.0, 0
    with torch.no_grad():
        for xb, yb in loader:
            prediction = model(xb.to(DEVICE))
            squared_error += nn.functional.mse_loss(
                prediction, yb.to(DEVICE), reduction="sum"
            ).item()
            count += len(xb)
    return math.sqrt(squared_error / count)


def fit(model, train_loader, val_loader, epochs=12, lr=0.008):
    model = model.to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history = {"train": [], "val": []}
    best_val, best_state = float("inf"), None
    for epoch in range(epochs):
        model.train()
        sum_sq_error, count = 0.0, 0
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            prediction = model(xb)
            loss = nn.functional.mse_loss(prediction, yb)
            # TODO: oblicz gradienty, ogranicz ich normę do 1.0, uaktualnij wagi.
            raise NotImplementedError("Uzupełnij kod: trzy kroki optymalizacji")
            sum_sq_error += loss.item() * len(xb)
            count += len(xb)
        val_rmse = evaluate(model, val_loader)
        history["train"].append(math.sqrt(sum_sq_error / count))
        history["val"].append(val_rmse)
        if val_rmse < best_val:
            best_val = val_rmse
            best_state = copy.deepcopy(model.state_dict())
    model.load_state_dict(best_state)
    return model, history


train_loader = DataLoader(TensorDataset(Xtr, ytr), batch_size=64, shuffle=True)
val_loader = DataLoader(TensorDataset(Xva, yva), batch_size=128, shuffle=False)
test_loader = DataLoader(TensorDataset(Xte, yte), batch_size=128, shuffle=False)


**Pytania do ćwiczenia 6**

1. Dlaczego ocena walidacyjna odbywa się bez `backward()` i `step()`?
2. Dlaczego warto zachować kopię parametrów z najlepszej epoki?


### Ćwiczenie 7. Porównanie trzech architektur

Uruchom trzy przebiegi uczenia. Przed uruchomieniem przewidź, czy najniższy błąd osiągnie model o największej liczbie parametrów. Na wykresie zaznaczamy **walidację**, a nie test. Zwycięzcę wybierzemy na podstawie najmniejszej wartości walidacyjnego RMSE; model po treningu otrzymał już najlepsze wagi z walidacji.

**Pytania:** 

Zmień `epochs` na 4 i na 20; czy ranking musi pozostać identyczny? Dlaczego pojedynczy wynik nie pozwala orzekać o wyższości całej rodziny modeli?


**Do uzupełnienia:** zapis wyników modelu do słownika.


In [ ]:
histories, trained, results = {}, {}, {}
for kind in ("RNN", "GRU", "LSTM"):
    torch.manual_seed(
        SEED
    )  # powtarzalna inicjalizacja (nie te same wagi dla różnych typów)
    model, history = fit(Forecaster(kind, 24), train_loader, val_loader, epochs=12)
    trained[kind], histories[kind] = model, history
    # TODO: zapisz do results liczbę parametrów i najniższy RMSE walidacji.
    raise NotImplementedError("Uzupełnij kod: słownik wyników modelu")
for kind, row in results.items():
    print(f'{kind:5} | parametry={row["parametry"]:5} | val RMSE={row["val_RMSE"]:.4f}')

plt.figure(figsize=(8.5, 3.3))
for kind, history in histories.items():
    plt.plot(range(1, len(history["val"]) + 1), history["val"], marker=".", label=kind)
plt.xlabel("Epoka")
plt.ylabel("RMSE walidacji (skala znormalizowana)")
plt.title("Ta sama procedura treningu, różne komórki rekurencyjne")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()


**Pytania do ćwiczenia 7**

1. Który model uzyskał najmniejszy walidacyjny RMSE?
2. Jakie dodatkowe doświadczenia byłyby potrzebne, aby wiarygodnie porównać architektury?


### Ćwiczenie 8. Ocena końcowa i porównanie z baselinami

Wybieramy architekturę **wyłącznie na podstawie walidacji**, a dopiero potem jednokrotnie oceniamy wybrany model na teście. Zestawimy go z dwiema metodami odniesienia: ostatnią wartością i sezonową prognozą z opóźnieniem 24. Wynik prognozy jednoetapowej zakłada, że w każdym kroku znamy **rzeczywistą** historię; nie oznacza prognozy bez dostępu do przyszłych pomiarów.

Przy powrocie do oryginalnych jednostek $\hat y_{\mathrm{org}}=\sigma\hat y_{\mathrm{scaled}}+\mu$. MAE i RMSE, jako miary różnic, mnożymy przez $\sigma$ (bez dodawania $\mu$).
**Do uzupełnienia:** wybór najlepszego modelu według walidacji.


In [ ]:
# TODO: wybierz model o najmniejszym val_RMSE ze słownika results.
raise NotImplementedError("Uzupełnij kod: wybór modelu na walidacji")
chosen = trained[winner]
chosen.eval()
with torch.no_grad():
    pred = chosen(Xte.to(DEVICE)).cpu()

print("Wybrany na walidacji:", winner)
for name, p in [
    ("Ostatnia wartość", baseline_te),
    ("Metoda sezonowa", seasonal_te),
    (f"Model {winner}", pred),
]:
    errors = metrics(p, yte)
    print(
        f'{name:20} | MAE={errors["MAE"]*sigma:.3f} | RMSE={errors["RMSE"]*sigma:.3f}'
    )

plt.figure(figsize=(11, 3.5))
period = slice(0, 120)
plt.plot(yte[period, 0].numpy() * sigma + mu, label="rzeczywistość", lw=2)
plt.plot(pred[period, 0].numpy() * sigma + mu, label=winner, alpha=0.9)
plt.plot(
    seasonal_te[period, 0].numpy() * sigma + mu, label="metoda sezonowa", alpha=0.65
)
plt.xlabel("Krok w okresie testowym")
plt.ylabel("Wartość oryginalna")
plt.title("Prognoza następnego kroku (wejście: prawdziwa historia)")
plt.legend()
plt.tight_layout()
plt.show()


**Pytania do ćwiczenia 8**

1. Czy wybrany model pokonał obie metody odniesienia na teście?
2. Dlaczego dobry wynik prognozy jednoetapowej nie przesądza o skuteczności prognozy wieloetapowej?


## 6. Dwukierunkowa sieć rekurencyjna (BiGRU)

### Dlaczego dwa kierunki?

Jednokierunkowa RNN buduje reprezentację pozycji z informacji po jej lewej stronie. W zadaniach takich jak etykietowanie słów w **całym znanym zdaniu** przydatny jest także kontekst po prawej stronie. **Sieć dwukierunkowa** przetwarza dostępną sekwencję niezależnie od początku i od końca, a następnie łączy reprezentacje obu kierunków. Dla stanu o wymiarze $H$ wynik ma wymiar $2H$.

W prognozowaniu szeregów czasowych nie wolno udostępniać modelowi przyszłych pomiarów. BiGRU może przetwarzać dwa kierunki **wewnątrz znanego okna historii**, ale nie może korzystać z próbek występujących po chwili prognozy. Dwukierunkowość nie oznacza więc automatycznie wycieku danych - o tym decyduje zakres wejścia.

Rozwiązanie opisali Schuster i Paliwal w 1997 r. [6]. Kosztem bogatszego kontekstu jest większa liczba parametrów i brak możliwości obliczenia reprezentacji pozycji na podstawie wyłącznie jej przeszłości.

### Ćwiczenie 9. Dwa kierunki i kształty stanów

Porównaj warstwy `GRU` oraz `BiGRU`. Ustal, gdzie w `h_n` znajdują się końcowe stany obu kierunków. Dlaczego `output[:, -1, :]` nie zawiera końcowego stanu kierunku wstecznego?

**Do uzupełnienia:** sklejenie dwóch stanów końcowych.


In [ ]:
xb = torch.randn(4, 12, 3)
uni = nn.GRU(3, 8, batch_first=True)
bi = nn.GRU(3, 8, batch_first=True, bidirectional=True)
out_u, h_u = uni(xb)
out_b, h_b = bi(xb)
print("GRU:", out_u.shape, h_u.shape, count_parameters(uni))
print("BiGRU:", out_b.shape, h_b.shape, count_parameters(bi))
assert out_u.shape == (4, 12, 8)
assert out_b.shape == (4, 12, 16)
assert h_b.shape == (2, 4, 8)
# TODO: połącz końcowe stany obu kierunków BiGRU wzdłuż ostatniej osi.
raise NotImplementedError("Uzupełnij kod: połączenie stanów BiGRU")
assert final_b.shape == (4, 16)


**Pytania do ćwiczenia 9**

1. Jak odczytać stany końcowe obu kierunków z `h_n`?
2. W jakich sytuacjach BiGRU mogłaby korzystać z informacji niedostępnej w chwili prognozy?


## 7. Połączenia *peephole* w LSTM

### Dlaczego bramki mają obserwować stan pamięci?

W zwykłej komórce LSTM bramki wejścia i zapominania zależą od bieżącego wejścia oraz poprzedniego stanu ukrytego. Stan komórki może jednak przenosić informację, która jest słabo widoczna w stanie ukrytym. **Połączenia *peephole*** dostarczają bramkom także informację o pamięci $c$, co bywa przydatne np. do rozpoznawania precyzyjnych odstępów czasowych.

W rozważanym wariancie bramki wejścia i zapominania korzystają z $c_{t-1}$, a bramka wyjścia z obliczonego już $c_t$. Wykorzystujemy trzy uczone wektory długości $H$, mnożone element po elemencie przez stan komórki. Przykład jest implementacją dydaktyczną pojedynczego kroku, a nie zoptymalizowaną warstwą do treningu całych sekwencji.

Opis i eksperymenty dotyczące *peephole* przedstawili Gers, Schraudolph i Schmidhuber w 2002 r. [7]. Ten wariant może pomagać w zadaniach wymagających dokładnego odmierzania czasu, ale nie jest uniwersalnie lepszy od standardowego LSTM.

### Ćwiczenie 10. Komórka LSTM z połączeniami *peephole*

Sprawdź działanie pojedynczego kroku, liczbę parametrów i propagację gradientu. Następnie porównaj wyjścia dla dwóch różnych wartości początkowego stanu komórki - przy ustalonych wejściu i stanie ukrytym.

**Do uzupełnienia:** aktualizacja stanu komórki LSTM.


In [ ]:
class PeepholeLSTMCell(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.linear = nn.Linear(input_size + hidden_size, 4 * hidden_size)
        self.p_i = nn.Parameter(torch.zeros(hidden_size))
        self.p_f = nn.Parameter(torch.zeros(hidden_size))
        self.p_o = nn.Parameter(torch.zeros(hidden_size))

    def forward(self, x, state):
        h_prev, c_prev = state
        i_raw, f_raw, g_raw, o_raw = self.linear(torch.cat([x, h_prev], dim=-1)).chunk(
            4, -1
        )
        i = torch.sigmoid(i_raw + self.p_i * c_prev)
        f = torch.sigmoid(f_raw + self.p_f * c_prev)
        g = torch.tanh(g_raw)
        # TODO: zapisz aktualizację pamięci c_new komórki LSTM.
        raise NotImplementedError("Uzupełnij kod: aktualizacja stanu komórki")
        o = torch.sigmoid(o_raw + self.p_o * c_new)
        h_new = o * torch.tanh(c_new)
        return h_new, c_new


cell = PeepholeLSTMCell(3, 8)
with torch.no_grad():
    cell.p_i.fill_(0.2)
    cell.p_f.fill_(0.3)
    cell.p_o.fill_(0.1)

x_ph = torch.randn(5, 3)
h_init = torch.zeros(5, 8)
c_zero, c_one = torch.zeros(5, 8), torch.ones(5, 8)
h0, c0 = cell(x_ph, (h_init, c_zero))
h1, c1 = cell(x_ph, (h_init, c_one))
assert h0.shape == h1.shape == c0.shape == c1.shape == (5, 8)
print("Peephole - różnica wyjść:", (h1 - h0).abs().mean().item())
print("Peephole (jeden bias i trzy wektory):", count_parameters(cell))
print("nn.LSTMCell (dwa biasy):", count_parameters(nn.LSTMCell(3, 8)))
assert count_parameters(cell) == count_parameters(nn.LSTMCell(3, 8)) - 8
(h1.square() + c1.square()).mean().backward()
assert cell.p_i.grad is not None and cell.p_i.grad.abs().sum() > 0


**Pytania do ćwiczenia 10**

1. Jaką dodatkową informację otrzymują bramki w wariancie *peephole*?
2. Skąd bierze się różnica liczby parametrów względem `nn.LSTMCell`?


## 8. Wielowarstwowa GRU i dropout

### Po co układać warstwy rekurencyjne jedna nad drugą?

Jedna warstwa GRU musi wydobywać wszystkie potrzebne zależności z przebiegu wejściowego. W **sieci wielowarstwowej** (*stacked GRU*) kolejna warstwa otrzymuje reprezentacje wyliczone przez poprzednią. Może dzięki temu modelować zależności na różnych poziomach złożoności, podobnie jak głębsze sieci z NM02. Zwiększa to jednak liczbę parametrów oraz ryzyko przeuczenia.

W PyTorch `dropout=p` w wielowarstwowym `nn.GRU` albo `nn.LSTM` działa między warstwami, nie na wyjściu warstwy ostatniej. Gdy `num_layers=1`, ten argument nie wprowadza maskowania pomiędzy warstwami. Tryb `train()` włącza działanie dropout, a `eval()` je wyłącza. `torch.no_grad()` to osobna operacja - wyłącza rejestrowanie gradientów, lecz nie zastępuje `eval()`.

### Ćwiczenie 11. Zachowanie dropout

Porównaj dwa przejścia tego samego wejścia w trybie `train()` i `eval()`. Sprawdź także, jak zmienia się liczba parametrów po dodaniu drugiej warstwy.

**Do uzupełnienia:** przełączenie w tryb ewaluacji.


In [ ]:
torch.manual_seed(SEED)
x_demo = torch.randn(3, 10, 2)
stacked = nn.GRU(2, 10, num_layers=2, dropout=0.5, batch_first=True)
stacked.train()
a = stacked(x_demo)[0]
b = stacked(x_demo)[0]
# TODO: przełącz sieć w tryb ewaluacji (wyłącz dropout).
raise NotImplementedError("Uzupełnij kod: tryb ewaluacji")
c = stacked(x_demo)[0]
d = stacked(x_demo)[0]
print("2 warstwy, trening: maks. różnica =", (a - b).abs().max().item())
print("2 warstwy, eval:    maks. różnica =", (c - d).abs().max().item())
assert torch.allclose(c, d)
assert count_parameters(nn.GRU(2, 10, num_layers=2)) > count_parameters(
    nn.GRU(2, 10, num_layers=1)
)


**Pytania do ćwiczenia 11**

1. Dlaczego w trybie `eval()` dwa wywołania dają ten sam wynik?
2. Dlaczego dropout zwiększa losowość uczenia, ale nie liczbę parametrów warstwy?


## 9. Minimal Gated Unit (MGU)

### Czy jedna bramka wystarczy?

Komórki LSTM i GRU wprowadzają kilka mechanizmów sterowania pamięcią. **MGU** (*Minimal Gated Unit*) jest próbą dalszego uproszczenia: wykorzystuje tylko jedną bramkę $f_t$, zarówno przy obliczaniu kandydata, jak i przy aktualizacji stanu. Motywacją jest ograniczenie liczby parametrów oraz uproszczenie analizy działania komórki.

Równania MGU w konwencji pracy źródłowej [8]:

$$\begin{aligned}
f_t&=\sigma(W_f[x_t,h_{t-1}]+b_f),\\
\widetilde h_t&=\tanh(W_g[x_t,f_t\odot h_{t-1}]+b_g),\\
h_t&=(1-f_t)\odot h_{t-1}+f_t\odot\widetilde h_t.
\end{aligned}$$

Gdy $f_t$ jest bliskie zera, przeważa stary stan; przy $f_t$ bliskim jedności przeważa nowy kandydat. MGU inaczej niż GRU wiąże bramkę z obliczaniem kandydata i aktualizacją pamięci. Mniejsza liczba parametrów nie musi oznaczać szybszego treningu; nasza implementacja wykonuje pętlę w Pythonie, a `nn.GRU` korzysta z wydajnych operacji bibliotecznych.

Komórkę MGU zaproponowali Zhou i inni w 2016 r. [8].

### Ćwiczenie 12. Komórka MGU od podstaw

Napisz komórkę z jedną bramką i jednym kandydatem, sprawdź rozmiary tensorów, gradienty oraz liczbę parametrów względem `nn.GRUCell`.

**Do uzupełnienia:** obliczenie kandydata MGU.


In [ ]:
class MGUCell(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.gate = nn.Linear(input_size + hidden_size, hidden_size)
        self.candidate = nn.Linear(input_size + hidden_size, hidden_size)

    def forward(self, x, h_prev):
        f = torch.sigmoid(self.gate(torch.cat([x, h_prev], dim=-1)))
        # TODO: kandydat MGU - połącz x z f*h_prev, przepuść przez candidate i tanh.
        raise NotImplementedError("Uzupełnij kod: kandydat komórki MGU")
        return (1 - f) * h_prev + f * candidate


class MGUForecaster(nn.Module):
    def __init__(self, hidden_size=24):
        super().__init__()
        self.cell = MGUCell(1, hidden_size)
        self.head = nn.Linear(hidden_size, 1)

    def forward(self, x):
        h = x.new_zeros(x.size(0), self.cell.gate.out_features)
        for t in range(x.size(1)):
            h = self.cell(x[:, t], h)
        return self.head(h)


mgu = MGUForecaster(24)
assert mgu(torch.randn(6, 36, 1)).shape == (6, 1)
mgu(torch.randn(6, 36, 1)).square().mean().backward()
assert mgu.cell.gate.weight.grad is not None
print("MGUCell:", count_parameters(MGUCell(1, 24)))
print("GRUCell:", count_parameters(nn.GRUCell(1, 24)))


**Pytania do ćwiczenia 12**

1. Jak zachowuje się MGU dla $f_t\approx0$ i $f_t\approx1$?
2. Dlaczego mniejsza liczba parametrów MGU nie gwarantuje krótszego czasu treningu?


## 10. Porównanie wariantów na tych samych danych

Dotąd rozpatrywaliśmy osobno dwukierunkowość, piętrowe warstwy i uproszczoną bramkę MGU. Teraz możemy porównać ich działanie na **tych samych przykładach walidacyjnych**. W BiGRU wykorzystamy końcowe stany obu kierunków; w dwuwarstwowej GRU - ostatni stan ostatniej warstwy; w MGU - stan po przejściu całego okna.

Wynik jest eksperymentem orientacyjnym. Wszystkie warianty otrzymują takie samo $H$ i budżet epok, ale różnią się liczbą parametrów i wydajnością wykonania. W tej sekcji nie korzystamy ze zbioru testowego.

### Ćwiczenie 13. Trzy odmiany sieci rekurencyjnych

Uruchom trening BiGRU, dwuwarstwowej GRU i MGU. Ustal ich liczbę parametrów i błąd walidacyjny. Wyniki potraktuj jako punkt wyjścia do dalszych doświadczeń, a nie jako ostateczny ranking.
**Do uzupełnienia:** złożenie reprezentacji BiGRU.


In [ ]:
class VariantForecaster(nn.Module):
    def __init__(self, variant, hidden_size=24):
        super().__init__()
        self.variant = variant
        if variant == "BiGRU":
            self.core = nn.GRU(1, hidden_size, batch_first=True, bidirectional=True)
            self.head = nn.Linear(2 * hidden_size, 1)
        elif variant == "StackedGRU":
            self.core = nn.GRU(1, hidden_size, num_layers=2, batch_first=True)
            self.head = nn.Linear(hidden_size, 1)
        elif variant == "MGU":
            self.core = MGUCell(1, hidden_size)
            self.head = nn.Linear(hidden_size, 1)
        else:
            raise ValueError(f"Nieznany wariant: {variant}")

    def forward(self, x):
        if self.variant == "MGU":
            h = x.new_zeros(x.size(0), self.core.gate.out_features)
            for step in range(x.size(1)):
                h = self.core(x[:, step, :], h)
            return self.head(h)
        _, h_n = self.core(x)
        if self.variant == "BiGRU":
            # TODO: reprezentacja BiGRU z końcowych stanów obu kierunków.
            raise NotImplementedError("Uzupełnij kod: reprezentacja BiGRU")
        else:
            representation = h_n[-1]
        return self.head(representation)


variant_results = {}
for variant in ("BiGRU", "StackedGRU", "MGU"):
    torch.manual_seed(SEED)
    model = VariantForecaster(variant, hidden_size=24)
    assert model(torch.zeros(3, LOOKBACK, 1)).shape == (3, 1)
    model, history = fit(model, train_loader, val_loader, epochs=8)
    variant_results[variant] = (count_parameters(model), min(history["val"]))
    print(
        f"{variant:10} | parametry={count_parameters(model):5} | "
        f"val RMSE={min(history['val']):.4f}"
    )


**Pytania do ćwiczenia 13**

1. Który wariant otrzymał najlepszy wynik na walidacji przy ustalonej liczbie epok?
2. Których porównań nie wolno interpretować jako dowodu przewagi architektury?


### Ćwiczenie 14. Stany ukryte w sieci dwukierunkowej

Dla jednopoziomowej BiGRU tensor `output` zawiera wyjścia z każdego punktu wejściowej sekwencji, natomiast `h_n` przechowuje stany końcowe obu przebiegów. Zwróć uwagę, że dla kierunku wstecznego **ostatnim przetworzonym** elementem jest pierwsza pozycja sekwencji.

Sprawdź równość właściwych fragmentów `output` i `h_n`. Podaj powód, dla którego samo `output[:, -1, :]` nie jest poprawnym podsumowaniem całej sekwencji w obu kierunkach.
**Do uzupełnienia:** sprawdzenie stanu kierunku wstecznego.


In [ ]:
probe = torch.randn(2, 7, 1)
check = nn.GRU(1, 5, bidirectional=True, batch_first=True)
output_bi, final_bi = check(probe)
assert output_bi.shape == (2, 7, 10)
assert final_bi.shape == (2, 2, 5)
assert torch.allclose(output_bi[:, -1, :5], final_bi[0], atol=1e-6)
# TODO: sprawdź równość końcowego stanu kierunku wstecznego i początku output.
raise NotImplementedError("Uzupełnij kod: asercja dla stanu wstecznego")
print("Koniec przebiegu w przód:", tuple(final_bi[0].shape))
print("Koniec przebiegu wstecz:", tuple(final_bi[1].shape))


**Pytania do ćwiczenia 14**

1. Które elementy `output` odpowiadają końcowym stanom obu kierunków?
2. Dlaczego kierunek wsteczny kończy przetwarzanie przy pierwszej pozycji oryginalnej sekwencji?


## 11. Prognozowanie wielu kroków

### Dlaczego prognoza wielu kroków jest trudniejsza?

W dotychczasowej ocenie model przy każdym kroku otrzymywał **rzeczywiste wcześniejsze obserwacje**. To prognozowanie jednoetapowe. Jeśli chcemy wyznaczyć 24 kolejne pomiary, nie znamy już rzeczywistych wartości po chwili rozpoczęcia prognozy. Jednym z rozwiązań jest **prognozowanie rekurencyjne**: przewidywana wartość trafia do okna jako nowa obserwacja wejściowa. Wówczas błędy kolejnych prognoz mogą się nawarstwiać.

Alternatywą jest model przewidujący cały horyzont za jednym razem (*direct multi-output*) albo osobne modele dla poszczególnych horyzontów. Nie będziemy ich tutaj trenować - chodzi o uchwycenie różnicy między dwoma sposobami oceny.

### Ćwiczenie 15. Prognoza rekurencyjna na 24 kroki

Wybierzmy ostatnie znane okno sprzed początku testu i wygenerujmy 24 prognozy bez wprowadzania rzeczywistych przyszłych próbek. Porównaj przebieg prognozy z odpowiadającym mu fragmentem szeregu testowego.
**Do uzupełnienia:** aktualizacja okna przy prognozowaniu.


In [ ]:
def forecast_recursive(model, context, horizon):
    if horizon < 1:
        raise ValueError("Horyzont prognozy musi być dodatni")
    model.eval()
    window = context.clone().reshape(1, -1, 1).to(DEVICE)
    predictions = []
    with torch.no_grad():
        for _ in range(horizon):
            value = model(window)  # (1, 1)
            predictions.append(value.squeeze().item())
            # TODO: przesuń okno o 1 krok i na koniec dopisz prognozę value.
            raise NotImplementedError("Uzupełnij kod: aktualizacja okna prognozy")
    return torch.tensor(predictions, dtype=torch.float32)


HORIZON = 24
# Pierwsze okno testowe kończy się w chwili 749; nie zawiera celu z chwili 750.
initial_context = Xte[0, :, 0]
recursive_scaled = forecast_recursive(chosen, initial_context, HORIZON)
actual_scaled = yte[:HORIZON, 0]
recursive_rmse = ((recursive_scaled - actual_scaled).square().mean().sqrt()).item()
print(
    f"RMSE jednej 24-krokowej ścieżki (jednostki oryginalne): "
    f"{recursive_rmse * sigma:.3f}"
)

plt.figure(figsize=(9, 3.4))
plt.plot(actual_scaled.numpy() * sigma + mu, marker="o", label="wartości rzeczywiste")
plt.plot(
    recursive_scaled.numpy() * sigma + mu, marker=".", label="prognoza rekurencyjna"
)
plt.xlabel("Horyzont prognozy (kroki)")
plt.ylabel("Wartość")
plt.title("24 kolejne kroki bez dostępu do nowych pomiarów")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()


**Pytania do ćwiczenia 15**

1. Jak zmienia się błąd, gdy prognozowane wartości stają się wejściem do następnych kroków?
2. Czym różni się sytuacja informacyjna prognozy jednoetapowej od rekurencyjnej?


## Bibliografia

**Prace źródłowe**

[1] Elman, J. L. (1990). *Finding Structure in Time*. **Cognitive Science, 14**(2), 179-211. https://doi.org/10.1207/s15516709cog1402_1

[2] Bengio, Y., Simard, P., Frasconi, P. (1994). *Learning Long-Term Dependencies with Gradient Descent Is Difficult*. **IEEE Transactions on Neural Networks, 5**(2), 157-166. https://doi.org/10.1109/72.279181

[3] Hochreiter, S., Schmidhuber, J. (1997). *Long Short-Term Memory*. **Neural Computation, 9**(8), 1735-1780. https://doi.org/10.1162/neco.1997.9.8.1735

[4] Gers, F. A., Schmidhuber, J., Cummins, F. (2000). *Learning to Forget: Continual Prediction with LSTM*. **Neural Computation, 12**(10), 2451-2471. https://doi.org/10.1162/089976600300015015

[5] Cho, K., van Merriënboer, B., Gulcehre, C., Bahdanau, D., Bougares, F., Schwenk, H., Bengio, Y. (2014). *Learning Phrase Representations using RNN Encoder-Decoder for Statistical Machine Translation*. **EMNLP 2014**, 1724-1734. https://aclanthology.org/D14-1179/

[6] Schuster, M., Paliwal, K. K. (1997). *Bidirectional Recurrent Neural Networks*. **IEEE Transactions on Signal Processing, 45**(11), 2673-2681. https://doi.org/10.1109/78.650093

[7] Gers, F. A., Schraudolph, N. N., Schmidhuber, J. (2002). *Learning Precise Timing with LSTM Recurrent Networks*. **Journal of Machine Learning Research, 3**, 115-143. https://jmlr.org/papers/v3/gers02a.html

[8] Zhou, G.-B., Wu, J., Zhang, C.-L., Zhou, Z.-H. (2016). *Minimal Gated Unit for Recurrent Neural Networks*. **arXiv:1603.09420**. https://arxiv.org/abs/1603.09420

**Dokumentacja i podręczniki**

[9] PyTorch, *RNN - dokumentacja warstwy*. https://docs.pytorch.org/docs/stable/generated/torch.nn.RNN.html

[10] PyTorch, *GRU - dokumentacja warstwy i opis kolejności działania bramki resetu*. https://docs.pytorch.org/docs/stable/generated/torch.nn.GRU.html

[11] PyTorch, *LSTM - dokumentacja warstwy*. https://docs.pytorch.org/docs/stable/generated/torch.nn.LSTM.html

[12] Hyndman, R. J., Athanasopoulos, G. (2021). *Forecasting: Principles and Practice* (wyd. 3). OTexts. https://otexts.com/fpp3/
